# Operate with OpenQASM

### References
- https://openqasm.com/versions/3.0/language/types.html
- https://quantum.cloud.ibm.com/docs/en/api/qiskit-runtime-rest
- https://quantum.cloud.ibm.com/docs/en/api/qiskit/qasm3
- https://quantum.cloud.ibm.com/docs/en/guides/interoperate-qiskit-qasm3

OpenQASM (open quantum assembly language), a machine-independent programming interface compatible with IBM® QPUs, is an imperative programming language for describing quantum circuits. OpenQASM uses the quantum circuit model to express quantum programs as ordered sequences of parameterized operations (such as gates, measurements, and resets) and real-time classical computation. In addition to quantum algorithms, OpenQASM can describe circuits intended to characterize, validate, or debug quantum processors.

In [2]:
# pip install qiskit-qasm3-import

Currently two high-level functions are available for importing from OpenQASM 3 into Qiskit. These functions are load(), which takes a file name, and loads(), which takes the program itself as a string:

In [9]:
import qiskit.qasm3
qiskit.qasm3.load(my_file.qasm) # Imports QASM3 file → QuantumCircuit
qiskit.qasm3.loads(program_string) # Imports QASM3 string → QuantumCircuit

We  can export Qiskit code to OpenQASM 3 with dumps(), which exports to a string, or dump(), which exports to a file.

In [4]:
from qiskit import QuantumCircuit
from qiskit.qasm3 import dumps

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

dumps(qc) # Exports QuantumCircuit → QASM3 string

'OPENQASM 3.0;\ninclude "stdgates.inc";\nbit[2] meas;\nqubit[2] q;\nh q[0];\ncx q[0], q[1];\nbarrier q[0], q[1];\nmeas[0] = measure q[0];\nmeas[1] = measure q[1];\n'

In [5]:
from qiskit import QuantumCircuit
from qiskit.qasm3 import dump

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

f = open("my_file.txt", "w")
dump(qc, f) # Exports QuantumCircuit → QASM3 file
f.close()

In [10]:
from qiskit import qasm3, QuantumCircuit, QuantumRegister, ClassicalRegister

# Build the circuit
qreg = QuantumRegister(3)
creg = ClassicalRegister(3)
qc = QuantumCircuit(qreg, creg)
with qc.switch(creg) as case:
    with case(0):
        qc.x(0)
    with case(1, 2):
        qc.x(1)
    with case(case.DEFAULT):
        qc.x(2)

# Export to an OpenQASM 3 string.
qasm_string = qasm3.dumps(qc, experimental=qasm3.ExperimentalFeatures.SWITCH_CASE_V1)

### All 4 High-Level Functions — Signatures

```python
# EXPORT (no extra install needed)
qiskit.qasm3.dumps(circuit, **kwargs)        # → str
qiskit.qasm3.dump(circuit, stream, **kwargs) # → None (writes to file/stream)

# IMPORT (requires: pip install qiskit[qasm3-import])
qiskit.qasm3.loads(program, *, num_qubits=None)   # → QuantumCircuit
qiskit.qasm3.load(filename, *, num_qubits=None)   # → QuantumCircuit
```

---

### Exceptions

| Exception | Raised by | When |
|---|---|---|
| `QASM3ExporterError` | `dump()`, `dumps()` | Export fails |
| `QASM3ImporterError` | `load()`, `loads()` | Import fails or circuit can't be represented |

```python
from qiskit.qasm3 import QASM3ExporterError, QASM3ImporterError
```

---

### `num_qubits` Parameter

`loads()` and `load()` accept an optional `num_qubits` keyword argument that tells the importer the total number of physical/virtual qubits in the target backend — so the result is full width even if the QASM program only uses a subset of qubits. 

```python
prog = '''
    OPENQASM 3.0;
    include "stdgates.inc";
    h $0;
    cx $0, $1;
'''
# Only uses 2 qubits, but backend has 5
qc = qasm3.loads(prog, num_qubits=5)
assert qc.num_qubits == 5  
```

---

### `Exporter` Class

> `dump()` and `dumps()` are single-use wrappers around `Exporter`.
> Use `Exporter` directly when exporting multiple circuits in one session.

```python
from qiskit.qasm3 import Exporter

exporter = Exporter(
    includes=('stdgates.inc',),   # default
    basis_gates=('U',),           # default
    disable_constants=False,      # False = use pi/euler/tau constants (default)
    indent='  ',                  # indentation string
)

qasm_str = exporter.dumps(qc)    # export to string
exporter.dump(qc, stream)        # export to file
```

#### Key `Exporter` parameter — `disable_constants`
- `False` (default) → values close to `pi`, `euler`, `tau` are emitted as those constants → **more accurate**
- `True` → always emit raw floating-point numbers

---

### Experimental Functions

Qiskit provides `load_experimental()` and `loads_experimental()` — a native Rust-based parser that is typically orders of magnitude faster, but has a reduced feature set and is still experimental. 

```python
# Experimental — faster but limited features
qiskit.qasm3.load_experimental(pathlike_or_filelike)
qiskit.qasm3.loads_experimental(source)

# Suppress the ExperimentalWarning if needed:
import warnings
from qiskit.exceptions import ExperimentalWarning
warnings.filterwarnings("ignore", category=ExperimentalWarning, module="qiskit.qasm3")
```

---

### `ExperimentalFeatures` Flag — `SWITCH_CASE_V1`

```python
# Use old switch-case syntax (pre-standardization):
qasm_string = qasm3.dumps(qc, experimental=qasm3.ExperimentalFeatures.SWITCH_CASE_V1)

# Combine multiple flags with |
qasm3.dumps(qc, experimental=flag1 | flag2)
```

| | Stabilized (default) | `SWITCH_CASE_V1` |
|---|---|---|
| Multiple case values | `case 0, 1 { }` | `case 0:\ncase 1:\n  ...\nbreak;` |

---

### Summary

```
qiskit.qasm3
├── dump(circuit, stream)         # Qiskit → QASM3 file   ← no extra install
├── dumps(circuit)                # Qiskit → QASM3 string ← no extra install
├── load(filename)                # QASM3 file → Qiskit   ← needs qasm3-import
├── loads(program)                # QASM3 string → Qiskit ← needs qasm3-import
├── load_experimental()           # faster, Rust-based, experimental
├── loads_experimental()          # faster, Rust-based, experimental
├── Exporter                      # class for multiple exports
├── QASM3ExporterError            # raised on export failure
└── QASM3ImporterError            # raised on import failure
```

## Qiskit Runtime REST API

> The REST API lets you submit jobs via HTTP requests instead of the Python SDK.
> Circuits are submitted as **OpenQASM strings**, not QuantumCircuit objects.

---

### Authentication — 3 Required Headers on EVERY Request

```python
headers = {
    "Authorization": "Bearer <YOUR_BEARER_TOKEN>",  # IAM token
    "Service-CRN":   "<YOUR_INSTANCE_CRN>",          # instance identifier
    "IBM-API-Version": "2026-02-15"                  # API version (YYYY-MM-DD)
}
```

### ⚠️ Bearer Token
- Expires after **max 1 hour**
- Generate from your API key via IAM
- Must generate a new one after expiry

---

### Submit a Job — Key Parameters

```python
payload = {
    "program_id": "estimator",   # "sampler" or "estimator" ← ONLY two options
    "backend":    "ibm_brisbane",
    "params": {
        "pubs": [[ "<OPENQASM_STRING>", "<OBSERVABLE>" ]],
        "version": 2,
        "options": { "dynamical_decoupling": {"enable": True} },
        "resilience_level": 1
    }
}
response = requests.post("https://quantum.cloud.ibm.com/api/v1/jobs",
                         headers=headers, json=payload)
```

### ⚠️ Key Facts
- `program_id` values: **`"sampler"`** or **`"estimator"`** only
- Circuits passed as **OpenQASM strings** (not QuantumCircuit objects)
- Multiple circuits → array of OpenQASM strings

---

### Session Workflow via REST

```python
# Step 1 — Create session
session_payload = { "mode": "dedicated", "max_ttl": 28800 }
response = requests.post(".../api/v1/sessions", headers=headers, json=session_payload)
session_id = response.json()['id']   # ← save this

# Step 2 — Submit jobs into that session
job_payload = {
    "program_id": "sampler",
    "backend":    "ibm_brisbane",
    "session_id": session_id,        # ← pass session ID here
    "params": { "pubs": [["<OPENQASM_STRING>"]], "version": 2 }
}
requests.post(".../api/v1/jobs", headers=headers, json=job_payload)
```

---

### REST API Endpoints

| Category | Key Operations |
|---|---|
| **Jobs** | Run, List, Get details, Cancel, Delete, Logs, Metrics, Results, Tags |
| **Backends** | List, Config, Properties, Status, Defaults |
| **Sessions** | Create, Get, Update, Close |
| **Instances** | Get details, Config, Usage |

---

### ⚠️ REST vs Python SDK — Key Difference

| | Python SDK | REST API |
|---|---|---|
| Circuit format | `QuantumCircuit` object | OpenQASM string |
| Primitive selection | `SamplerV2()` / `EstimatorV2()` | `"program_id": "sampler"/"estimator"` |
| Auth | API key in `QiskitRuntimeService()` | Bearer token in HTTP header |
| Session | `Session(backend=...)` | POST to `/api/v1/sessions` → use returned `id` |

## OpenQASM 3.0 — Types Cheat Sheet

| Type | One-liner |
|---|---|
| `qubit` / `qubit[n]` | quantum bit / register |
| `bit` / `bit[n]` | classical bit / measurement results |
| `int[n]` / `uint[n]` | signed / unsigned integer |
| `float[n]` | floating point |
| `bool` | true / false |
| `angle[n]` | efficient angle representation, modulo 2π |
| `duration` | fixed timing |
| `stretch` | variable timing, resolved at compile time |

### ⚠️ 3 Rules to Remember
1. `stretch` and `bit` — **NOT valid** as array base types
2. No comma-separated declarations — `int x, y;` ❌ — one at a time only
3. Qubit register size must be a **compile-time constant**